# 02 — Render and report

Run this notebook after training and quantitative evaluation. Rendering replays fixed manifest rows with a photorealistic chase camera; no video is used as a PPO observation.

## SECTION 0 — User settings

In [ ]:
REPO_URL = "REPLACE_WITH_GITHUB_URL"
REPO_BRANCH = "main"
REPO_DIR = "/content/carla-highway-rl"
DRIVE_ROOT = "/content/drive/MyDrive/CARLA_Highway_RL"
CARLA_SERVER_MODE = "external"
CARLA_HOST = "127.0.0.1"
CARLA_PORT = 2000
CARLA_TM_PORT = 8000
CARLA_ROOT = ""
RUN_NAME = "ppo_seed_0"
SEED = 0

## SECTION 1 — Common initialization

This repeats notebook 1's safe initialization so every later section works in a fresh runtime. The requirements accept the runtime's existing NumPy 2 release instead of downgrading it. The cell checks for a real loaded-versus-installed version mismatch and validates all binary imports in a fresh subprocess before continuing.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
repo = Path(REPO_DIR)
if not repo.exists():
    if REPO_URL == "REPLACE_WITH_GITHUB_URL":
        raise RuntimeError("Set REPO_URL before cloning in a hosted runtime.")
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(repo)], check=True)
else:
    status = subprocess.run(["git", "status", "--porcelain"], cwd=repo, check=True, capture_output=True, text=True)
    print(status.stdout or "Working tree clean.")
    if not status.stdout.strip():
        subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=repo, check=True)
        subprocess.run(["git", "merge", "--ff-only", "FETCH_HEAD"], cwd=repo, check=True)
    else:
        print("Local edits preserved; automatic update skipped.")
os.chdir(repo)
os.environ.update({"CARLA_HOST": CARLA_HOST, "CARLA_PORT": str(CARLA_PORT), "CARLA_TM_PORT": str(CARLA_TM_PORT), "CARLA_ROOT": CARLA_ROOT, "CARLA_SERVER_MODE": CARLA_SERVER_MODE, "HIGHWAY_RL_ARTIFACT_ROOT": str(repo / "artifacts"), "HIGHWAY_RL_DRIVE_ROOT": DRIVE_ROOT})
print("Repository:", repo)
print("Artifacts:", os.environ["HIGHWAY_RL_ARTIFACT_ROOT"])
print("Drive:", DRIVE_ROOT)
numpy_loaded = sys.modules.get("numpy")
numpy_loaded_version = getattr(numpy_loaded, "__version__", None)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
import importlib.metadata
numpy_installed_version = importlib.metadata.version("numpy")
if numpy_loaded_version and numpy_loaded_version != numpy_installed_version:
    raise RuntimeError(f"NumPy changed from loaded {numpy_loaded_version} to installed {numpy_installed_version}. Restart this runtime once, then rerun Sections 0–1.")
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
subprocess.run([sys.executable, "-c", "import carla, cv2, gymnasium, matplotlib, numpy, pandas, stable_baselines3, torch, yaml; print('Dependency imports OK; NumPy', numpy.__version__)"], check=True)
for package in ["carla", "numpy", "gymnasium", "stable-baselines3", "pandas", "matplotlib", "PyYAML", "torch", "opencv-python-headless"]:
    try:
        print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        print(package, "not installed")
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--from-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)
subprocess.run([sys.executable, "run.py", "validate-config", "--config", "config.yaml"], check=True)

## SECTION 2 — Rendering server setup

RGB sensors require `world.no_rendering_mode=false`. Managed mode is started with Epic quality and off-screen rendering; external mode must already expose a rendering-enabled CARLA 0.9.16 server. The final command captures and validates one synchronized RGB frame batch.

In [ ]:
manifest = Path("artifacts/manifests/evaluation_manifest.json")
if not manifest.exists():
    raise FileNotFoundError("Restore or generate the frozen evaluation manifest first.")
if CARLA_SERVER_MODE == "managed":
    subprocess.run([sys.executable, "run.py", "server", "stop", "--config", "config.yaml"], check=False)
    subprocess.run([sys.executable, "run.py", "server", "start", "--config", "config.yaml", "--rendering"], check=True)
subprocess.run([sys.executable, "run.py", "doctor", "--config", "config.yaml"], check=True)
subprocess.run([sys.executable, "run.py", "smoke", "--config", "config.yaml", "--render-frame-only", "--manifest", str(manifest)], check=True)
print(Path("artifacts/logs/runtime/render_frame_smoke.json").read_text())

## SECTION 3 — Load evaluation results

In [ ]:
import json, hashlib
import pandas as pd
from IPython.display import display
episodes = pd.read_csv("artifacts/evaluations/episode_results.csv")
summary = pd.read_csv("artifacts/evaluations/summary_results.csv")
display(summary[summary["group_type"] == "overall"])
manifest_data = json.loads(manifest.read_text())
model = Path("artifacts/models") / RUN_NAME / "final_model.zip"
if not model.exists():
    raise FileNotFoundError(model)
if set(episodes["manifest_hash"].dropna()) != {manifest_data["manifest_hash"]}:
    raise RuntimeError("Evaluation CSV and manifest hashes disagree.")
print("Manifest:", manifest_data["manifest_hash"])
model_hash = hashlib.sha256(model.read_bytes()).hexdigest()
training_metadata = json.loads((Path("artifacts/models") / RUN_NAME / "training_metadata.json").read_text())
if training_metadata.get("final_model_sha256") != model_hash:
    raise RuntimeError("Model hash does not match its saved training metadata.")
print("Model SHA-256:", model_hash)

## SECTION 4 — Automatic video selection

No category is fabricated. Empty categories remain absent unless `--allow-fallback` is explicitly added.

In [ ]:
subprocess.run([sys.executable, "run.py", "select-videos", "--config", "config.yaml", "--episodes", "artifacts/evaluations/episode_results.csv", "--model", str(model)], check=True)
video_manifest_path = Path("artifacts/manifests/video_manifest.json")
video_manifest = json.loads(video_manifest_path.read_text())
display(pd.DataFrame(video_manifest["selections"]))

### Optional explicit condition overrides

Edit only the desired category mappings. Empty means keep automatic selection.

In [ ]:
CONDITION_OVERRIDES = {
    "ppo_safe_overtake": "",
    "ppo_rejected_unsafe_lane_change": "",
    "ppo_dense_traffic_success": "",
    "keep_lane_blocked": "",
    "random_failure": "",
    "ppo_failure": "",
}
valid_conditions = set(episodes["condition_id"])
for selection in video_manifest["selections"]:
    override = CONDITION_OVERRIDES.get(selection["category"], "")
    if override:
        if override not in valid_conditions:
            raise ValueError(f"Unknown condition override: {override}")
        selection["condition_id"] = override
        selection["reason_selected"] = "Explicit notebook override."
video_manifest_path.write_text(json.dumps(video_manifest, indent=2) + "\n")
display(pd.DataFrame(video_manifest["selections"]))

## SECTION 5 — Render six videos

Each selected category is independently resumable. After every completed category, artifacts are synchronized to Drive.

In [ ]:
for selection in video_manifest["selections"]:
    category = selection["category"]
    subprocess.run([sys.executable, "run.py", "render-videos", "--config", "config.yaml", "--manifest", str(manifest), "--video-manifest", str(video_manifest_path), "--model", str(model), "--category", category, "--resume-existing"], check=True)
    subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)
for video in sorted(Path("artifacts/videos").glob("*.mp4")):
    print(video.name, video.stat().st_size, "bytes")

## SECTION 6 — Display videos

In [ ]:
from IPython.display import Video, Markdown
video_manifest = json.loads(video_manifest_path.read_text())
for item in video_manifest["selections"]:
    path = Path(item.get("mp4_path", ""))
    if path.exists():
        display(Markdown(f"### {item['category']} — {item['condition_id']} — {item.get('outcome', '')}"))
        display(Video(str(path), embed=True))

## SECTION 7 — Generate final report data

In [ ]:
subprocess.run([sys.executable, "run.py", "report-data", "--config", "config.yaml"], check=True)
for path in [Path("reports/generated/report_values.tex"), Path("reports/generated/results_table.tex"), Path("reports/generated/paired_table.tex")]:
    print("\n---", path, "---\n", path.read_text())

## SECTION 8 — Compile reports when LaTeX exists

In [ ]:
import shutil
if shutil.which("pdflatex"):
    for report in ["reports/main_report.tex", "reports/technical_log.tex"]:
        for _ in range(2):
            subprocess.run(["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "-output-directory", "reports", report], check=True)
else:
    print("pdflatex is unavailable; source and generated inputs are preserved without failing the notebook.")
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)

## SECTION 9 — Final artifact inventory

In [ ]:
for pattern in ["artifacts/models/**/*.zip", "artifacts/evaluations/*.csv", "artifacts/plots/*.png", "artifacts/videos/*.mp4", "artifacts/recordings/*", "reports/*.pdf"]:
    print("\n", pattern)
    for path in sorted(Path(".").glob(pattern)):
        print(path, path.stat().st_size)
subprocess.run([sys.executable, "run.py", "sync", "--config", "config.yaml", "--to-drive", "--local-root", "artifacts", "--drive-root", DRIVE_ROOT], check=True)
if CARLA_SERVER_MODE == "managed":
    subprocess.run([sys.executable, "run.py", "server", "stop", "--config", "config.yaml"], check=True)
else:
    print("External CARLA server left untouched.")
print("Drive root:", DRIVE_ROOT)